# 20.1 端侧检索与向量库

Mini embedding + 暴力/HNSW 风格简化检索 + 上下文预算裁剪。

In [ ]:
import hashlib
import json
import math
import time
from dataclasses import dataclass, field
from typing import Optional
import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    torch.manual_seed(42)
    print(f"PyTorch {torch.__version__}")
except Exception as e:
    torch = None
    print("torch unavailable:", e)

np.random.seed(42)

In [ ]:
def embed(text: str, dim=64) -> np.ndarray:
    v = np.zeros(dim, dtype=np.float32)
    for ch in text.lower():
        v[hash(ch) % dim] += 1.0
    n = np.linalg.norm(v) + 1e-8
    return v / n


docs = [
    "出差报销需要发票和行程单",
    "端侧模型推荐 INT4 量化与 KV 压缩",
    "电池低于百分之二十应切换小模型",
    "车载语音助手不能直连刹车控制",
    "OTA 更新应支持断点续传与签名校验",
]
mat = np.stack([embed(d) for d in docs])


def brute_topk(query: str, k=2):
    q = embed(query)
    scores = mat @ q
    idx = np.argsort(-scores)[:k]
    return [(docs[i], float(scores[i])) for i in idx]


print(brute_topk("手机电量低如何部署"))

In [ ]:
@dataclass
class HNSWLite:
    """教学用极简近邻图：不是真 HNSW，只演示图检索思路。"""
    vectors: np.ndarray
    edges: list
    ef: int = 8

    @classmethod
    def build(cls, vectors, degree=4):
        n = len(vectors)
        edges = [set() for _ in range(n)]
        for i in range(n):
            sims = vectors @ vectors[i]
            sims[i] = -1
            nbrs = np.argsort(-sims)[:degree]
            for j in nbrs:
                edges[i].add(int(j)); edges[int(j)].add(i)
        return cls(vectors, [list(e) for e in edges])

    def search(self, q, k=2):
        # 从 0 号点贪心扩展
        entry = 0
        visited = set()
        frontier = [entry]
        best = []
        while frontier:
            i = frontier.pop()
            if i in visited:
                continue
            visited.add(i)
            s = float(self.vectors[i] @ q)
            best.append((s, i))
            if len(visited) < self.ef:
                frontier.extend(self.edges[i])
        best.sort(reverse=True)
        return best[:k]


index = HNSWLite.build(mat)
q = embed("OTA 签名与续传")
hits = index.search(q, k=2)
print("hnsw-lite", [(docs[i], round(s,3)) for s,i in hits])


def fit_context(snippets, token_budget=24):
    used = 0; out = []
    for s in snippets:
        t = max(1, len(s)//2)
        if used + t > token_budget:
            break
        out.append(s); used += t
    return out, used

snips = [docs[i] for _,i in hits]
print("context", fit_context(snips, 20))

## 小结

端侧检索=小 Embedding + 节制索引 + 严格上下文预算；语料小用暴力，中等上 HNSW/sqlite-vec。